In [1]:
from pathlib import Path
from tqdm import tqdm
import pydicom
import re
import shutil
import pandas as pd
import SimpleITK as sitk


# Extract relevant series directories

In [31]:
def get_dirs(directories: set, path: Path):
    if 'IM' in path.name:
        directories.add(path.parents[0])
    else:
        for subdir in path.iterdir():
            get_dirs(directories, subdir)

In [32]:
d = set()
p = Path('/home/simon/Data/Augsburg_large/patients/')

In [33]:
get_dirs(d, p)

In [34]:
r = re.compile(r'PA\d+')

In [37]:
def create_dupliacte_dir(path: str, no: int) -> str:
    try:
        Path(f'{path}_{no}').mkdir(parents=True, exist_ok=False)
        return f'{path}_{no}'
    except FileExistsError:
        print(f'{path}_{no}: Directory already exists. Trying next number.')
        return create_dupliacte_dir(path, no + 1)

In [39]:
for i in tqdm(d):
    patient = r.search(str(i))[0]  # get patient identifier
    dataset = pydicom.dcmread(f'{i}/IM000001')
    
    if (dataset[0x0008,0x103e].value == 't1_tse_tra Hüften bds') or (dataset[0x0008,0x103e].value == 't1_tse_tra Hüften bds 10mm'):
        try:
            Path(f'/home/simon/Data/Augsburg_large/preprocessed/{patient}/hip').mkdir(parents=True, exist_ok=False)
            new_path = f'/home/simon/Data/Augsburg_large/preprocessed/{patient}/hip'
        except FileExistsError:
            new_path = create_dupliacte_dir(f'/home/simon/Data/Augsburg_large/preprocessed/{patient}/hip', 1)
            # print(f'{patient}/{i}: Duplicate series found. Moving into {new_path}')
        for image in i.iterdir():
            shutil.copy(image, new_path)
    elif dataset[0x0008,0x103e].value == 't1_tse_tra OSG':
        try:
            Path(f'/home/simon/Data/Augsburg_large/preprocessed/{patient}/ankle').mkdir(parents=True, exist_ok=False)
            new_path = f'/home/simon/Data/Augsburg_large/preprocessed/{patient}/ankle'
        except FileExistsError:
            new_path = create_dupliacte_dir(f'/home/simon/Data/Augsburg_large/preprocessed/{patient}/ankle', 1)
            # print(f'{patient}/{i}: Duplicate series found. Moving into {new_path}')
        for image in i.iterdir():
            shutil.copy(image, new_path)
    elif dataset[0x0008,0x103e].value == 't1_tse_tra Knie':
        try:
            Path(f'/home/simon/Data/Augsburg_large/preprocessed/{patient}/knee').mkdir(parents=True, exist_ok=False)
            new_path = f'/home/simon/Data/Augsburg_large/preprocessed/{patient}/knee'
        except FileExistsError:
            new_path = create_dupliacte_dir(f'/home/simon/Data/Augsburg_large/preprocessed/{patient}/knee', 1)
            # print(f'{patient}/{i}: Duplicate series found. Moving into {new_path}')
        for image in i.iterdir():
            shutil.copy(image, new_path)
    else:
        continue

100%|██████████| 9363/9363 [00:10<00:00, 930.40it/s] 


# DICOM Metadata exploration

In [55]:
patients = [x.name for x in Path('/home/simon/Data/Augsburg_large/preprocessed/').iterdir()]
index = pd.MultiIndex.from_product([patients, ['hip', 'ankle', 'knee']], names=['Patient', 'BodyPart'])
df = pd.DataFrame(columns=['SliceThickness', 'PixelSpacing', 'EchoTime', 'RepetitionTime', 'MagneticFieldStrength', 'Manufacturer', 'ManufacturerModelName', 'NumSlices'], index=index)

In [56]:
df

SliceThickness PixelSpacing EchoTime RepetitionTime  \
Patient  BodyPart                                                       
PA000291 hip                 NaN          NaN      NaN            NaN   
         ankle               NaN          NaN      NaN            NaN   
         knee                NaN          NaN      NaN            NaN   
PA000464 hip                 NaN          NaN      NaN            NaN   
         ankle               NaN          NaN      NaN            NaN   
...                          ...          ...      ...            ...   
PA000807 ankle               NaN          NaN      NaN            NaN   
         knee                NaN          NaN      NaN            NaN   
PA000227 hip                 NaN          NaN      NaN            NaN   
         ankle               NaN          NaN      NaN            NaN   
         knee                NaN          NaN      NaN            NaN   

                  MagneticFieldStrength Manufacturer ManufacturerModelName  \
Patient  BodyPart                                                            
PA000291 hip                        NaN          NaN                   NaN   
         ankle                      NaN          NaN                   NaN   
         knee                       NaN          NaN                   NaN   
PA000464 hip                        NaN          NaN                   NaN   
         ankle                      NaN          NaN                   NaN   
...                                 ...          ...                   ...   
PA000807 ankle                      NaN          NaN                   NaN   
         knee                       NaN          NaN                   NaN   
PA000227 hip                        NaN          NaN                   NaN   
         ankle                      NaN          NaN                   NaN   
         knee                       NaN          NaN                   NaN   

                  NumSlices  
Patient  BodyPart            
PA000291 hip            NaN  
         ankle          NaN  
         knee           NaN  
PA000464 hip            NaN  
         ankle          NaN  
...                     ...  
PA000807 ankle          NaN  
         knee           NaN  
PA000227 hip            NaN  
         ankle          NaN  
         knee           NaN  

[2862 rows x 8 columns]

In [57]:
for patient in patients:
    for bodypart in ['hip', 'ankle', 'knee']:
        path = Path(f'/home/simon/Data/Augsburg_large/preprocessed/{patient}/{bodypart}/')
        try:
            dataset = pydicom.dcmread(path / 'IM000001')
        except FileNotFoundError:
            continue
            
        df.loc[(patient, bodypart), 'SliceThickness'] = dataset[0x0018,0x0050].value
        df.loc[(patient, bodypart), 'PixelSpacing'] = dataset[0x0028,0x0030].value
        df.loc[(patient, bodypart), 'EchoTime'] = dataset[0x0018,0x0081].value
        df.loc[(patient, bodypart), 'RepetitionTime'] = dataset[0x0018,0x0080].value
        df.loc[(patient, bodypart), 'MagneticFieldStrength'] = dataset[0x0018,0x0087].value
        df.loc[(patient, bodypart), 'Manufacturer'] = dataset[0x0008,0x0070].value
        df.loc[(patient, bodypart), 'ManufacturerModelName'] = dataset[0x0008,0x1090].value
        df.loc[(patient, bodypart), 'NumSlices'] = len(list(path.iterdir()))

In [67]:
df = df.dropna(axis=0)
df = df.astype({'SliceThickness': int, 'EchoTime': float, 'RepetitionTime': float, 'MagneticFieldStrength': float, 'NumSlices': int})

In [68]:
df

SliceThickness        PixelSpacing  EchoTime  \
Patient  BodyPart                                                 
PA000291 hip                   10  [1.09375, 1.09375]      11.0   
         ankle                  4  [1.09375, 1.09375]      11.0   
         knee                   4  [1.09375, 1.09375]      11.0   
PA000464 hip                   10  [1.09375, 1.09375]      11.0   
         ankle                  4  [1.09375, 1.09375]      11.0   
...                           ...                 ...       ...   
PA000807 ankle                  4  [1.09375, 1.09375]      11.0   
         knee                   4  [1.09375, 1.09375]      11.0   
PA000227 hip                   10  [1.09375, 1.09375]      11.0   
         ankle                  4  [1.09375, 1.09375]      11.0   
         knee                   4  [1.09375, 1.09375]      11.0   

                   RepetitionTime  MagneticFieldStrength Manufacturer  \
Patient  BodyPart                                                       
PA000291 hip                671.0                    1.5      SIEMENS   
         ankle              503.0                    1.5      SIEMENS   
         knee               503.0                    1.5      SIEMENS   
PA000464 hip                629.0                    1.5      SIEMENS   
         ankle              503.0                    1.5      SIEMENS   
...                           ...                    ...          ...   
PA000807 ankle              503.0                    1.5      SIEMENS   
         knee               503.0                    1.5      SIEMENS   
PA000227 hip                629.0                    1.5      SIEMENS   
         ankle              503.0                    1.5      SIEMENS   
         knee               503.0                    1.5      SIEMENS   

                  ManufacturerModelName  NumSlices  
Patient  BodyPart                                   
PA000291 hip                     Avanto         16  
         ankle                   Avanto         24  
         knee                    Avanto         26  
PA000464 hip                     Avanto         30  
         ankle                   Avanto         30  
...                                 ...        ...  
PA000807 ankle                   Avanto         24  
         knee                    Avanto         26  
PA000227 hip                     Avanto         30  
         ankle                   Avanto         30  
         knee                    Avanto         34  

[2856 rows x 8 columns]

In [77]:
df.info()

<class 'pandas.core.frame.DataFrame'>
MultiIndex: 2856 entries, ('PA000291', 'hip') to ('PA000227', 'knee')
Data columns (total 8 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   SliceThickness         2856 non-null   int64  
 1   PixelSpacing           2856 non-null   object 
 2   EchoTime               2856 non-null   float64
 3   RepetitionTime         2856 non-null   float64
 4   MagneticFieldStrength  2856 non-null   float64
 5   Manufacturer           2856 non-null   object 
 6   ManufacturerModelName  2856 non-null   object 
 7   NumSlices              2856 non-null   int64  
dtypes: float64(3), int64(2), object(3)
memory usage: 226.8+ KB


## Summary statistics

### Hip

In [76]:
df[['SliceThickness', 'EchoTime', 'RepetitionTime', 'MagneticFieldStrength', 'NumSlices']].swaplevel(0, 1, axis=0).loc['hip'].describe()

,SliceThickness,EchoTime,RepetitionTime,MagneticFieldStrength,NumSlices
count,951.000000,951.000000,951.000000,951.0,951.000000
mean,9.988433,10.998002,655.720294,1.5,19.681388
std,0.202284,0.061612,38.054000,0.0,6.221225
min,5.000000,9.100000,420.000000,1.5,12.000000
25%,10.000000,11.000000,629.000000,1.5,16.000000
50%,10.000000,11.000000,671.000000,1.5,16.000000
75%,10.000000,11.000000,671.000000,1.5,30.000000
max,10.000000,11.000000,965.000000,1.5,33.000000


### Knee

In [78]:
df[['SliceThickness', 'EchoTime', 'RepetitionTime', 'MagneticFieldStrength', 'NumSlices']].swaplevel(0, 1, axis=0).loc['knee'].describe()

,SliceThickness,EchoTime,RepetitionTime,MagneticFieldStrength,NumSlices
count,952.0,952.0,952.000000,952.0,952.000000
mean,4.0,11.0,526.903361,1.5,30.382353
std,0.0,0.0,29.724588,0.0,4.615480
min,4.0,11.0,461.000000,1.5,24.000000
25%,4.0,11.0,503.000000,1.5,26.000000
50%,4.0,11.0,545.000000,1.5,26.000000
75%,4.0,11.0,545.000000,1.5,34.000000
max,4.0,11.0,1090.000000,1.5,44.000000


### Ankle

In [79]:
df[['SliceThickness', 'EchoTime', 'RepetitionTime', 'MagneticFieldStrength', 'NumSlices']].swaplevel(0, 1, axis=0).loc['ankle'].describe()

,SliceThickness,EchoTime,RepetitionTime,MagneticFieldStrength,NumSlices
count,953.0,953.0,953.000000,953.0,953.000000
mean,4.0,11.0,503.500525,1.5,27.284365
std,0.0,0.0,21.068871,0.0,3.632535
min,4.0,11.0,419.000000,1.5,12.000000
25%,4.0,11.0,503.000000,1.5,24.000000
50%,4.0,11.0,503.000000,1.5,24.000000
75%,4.0,11.0,503.000000,1.5,30.000000
max,4.0,11.0,671.000000,1.5,36.000000


### Pixel Spacing & Manufacturer

In [70]:
df[['PixelSpacing', 'Manufacturer', 'ManufacturerModelName']].describe()

,PixelSpacing,Manufacturer,ManufacturerModelName
count,2856,2856,2856
unique,5,1,1
top,"[1.09375, 1.09375]",SIEMENS,Avanto
freq,2850,2856,2856


In [72]:
df['PixelSpacing'].value_counts()

PixelSpacing
[1.09375, 1.09375]                    2850
[1.1458333730698, 1.1458333730698]       3
[1.25, 1.25]                             1
[1.3020833730698, 1.3020833730698]       1
[1.640625, 1.640625]                     1
Name: count, dtype: int64

# Nifti Conversion

In [81]:
from scripts.dicom_to_nifti import dicom_to_nifti

In [84]:
for patient in Path('/home/simon/Data/Augsburg_large/preprocessed/').iterdir():
    for bodypart in ['hip', 'ankle', 'knee']:
        try:
            dicom_to_nifti(f'/home/simon/Data/Augsburg_large/preprocessed/{patient.name}/{bodypart}', f'/home/simon/Data/Augsburg_large/preprocessed/{patient.name}/{bodypart}.nii.gz')
        except RuntimeError as e:
            print(f'Conversion failed for {patient}', e)
            continue

Conversion failed for /home/simon/Data/Augsburg_large/preprocessed/PA000964 Exception thrown in SimpleITK ImageSeriesReader_Execute: /tmp/SimpleITK/Code/IO/src/sitkImageSeriesReader.cxx:131:
sitk::ERROR: File names information is empty. Cannot read series.
Conversion failed for /home/simon/Data/Augsburg_large/preprocessed/PA000964 Exception thrown in SimpleITK ImageSeriesReader_Execute: /tmp/SimpleITK/Code/IO/src/sitkImageSeriesReader.cxx:131:
sitk::ERROR: File names information is empty. Cannot read series.


GDCMSeriesFileNames (0x66baa50): /home/simon/Data/Augsburg_large/preprocessed/PA000964/hip is not a directory

GDCMSeriesFileNames (0x66baa50): No Series can be found, make sure your restrictions are not too strong

GDCMSeriesFileNames (0x66baa50): /home/simon/Data/Augsburg_large/preprocessed/PA000964/knee is not a directory

GDCMSeriesFileNames (0x66baa50): No Series can be found, make sure your restrictions are not too strong



Conversion failed for /home/simon/Data/Augsburg_large/preprocessed/PA000917 Exception thrown in SimpleITK ImageSeriesReader_Execute: /tmp/SimpleITK/Code/IO/src/sitkImageSeriesReader.cxx:131:
sitk::ERROR: File names information is empty. Cannot read series.


GDCMSeriesFileNames (0x66baa50): /home/simon/Data/Augsburg_large/preprocessed/PA000917/hip is not a directory

GDCMSeriesFileNames (0x66baa50): No Series can be found, make sure your restrictions are not too strong



Conversion failed for /home/simon/Data/Augsburg_large/preprocessed/PA000916 Exception thrown in SimpleITK ImageSeriesReader_Execute: /tmp/SimpleITK/Code/IO/src/sitkImageSeriesReader.cxx:131:
sitk::ERROR: File names information is empty. Cannot read series.


GDCMSeriesFileNames (0x66baa50): /home/simon/Data/Augsburg_large/preprocessed/PA000916/hip is not a directory

GDCMSeriesFileNames (0x66baa50): No Series can be found, make sure your restrictions are not too strong

GDCMSeriesFileNames (0x66baa50): /home/simon/Data/Augsburg_large/preprocessed/PA000255/ankle is not a directory

GDCMSeriesFileNames (0x66baa50): No Series can be found, make sure your restrictions are not too strong

GDCMSeriesFileNames (0x66baa50): /home/simon/Data/Augsburg_large/preprocessed/PA000255/knee is not a directory

GDCMSeriesFileNames (0x66baa50): No Series can be found, make sure your restrictions are not too strong



Conversion failed for /home/simon/Data/Augsburg_large/preprocessed/PA000255 Exception thrown in SimpleITK ImageSeriesReader_Execute: /tmp/SimpleITK/Code/IO/src/sitkImageSeriesReader.cxx:131:
sitk::ERROR: File names information is empty. Cannot read series.
Conversion failed for /home/simon/Data/Augsburg_large/preprocessed/PA000255 Exception thrown in SimpleITK ImageSeriesReader_Execute: /tmp/SimpleITK/Code/IO/src/sitkImageSeriesReader.cxx:131:
sitk::ERROR: File names information is empty. Cannot read series.


# Patient information

In [6]:
patients = [x.name for x in Path('/home/simon/Data/Augsburg_large/preprocessed/').iterdir()]
index = pd.Index(patients, name='Patient')
df = pd.DataFrame(columns=['Age', 'Sex'], index=index)
for patient in tqdm(Path('/home/simon/Data/Augsburg_large/preprocessed/').iterdir()):
    try:
        ds = pydicom.dcmread(patient / 'hip/IM000001')
    except FileNotFoundError as e:
        df.loc[patient.name, 'Age'] = None
        df.loc[patient.name, 'Sex'] = None
        print(e)
        continue

    df.loc[patient.name, 'Sex'] = ds[0x0010,0x0040].value
    dob = ds[0x0010,0x0030].value
    study_date = ds[0x0008,0x0020].value
    age = int(study_date[:4]) - int(dob[:4])
    df.loc[patient.name, 'Age'] = age

df.to_csv('/home/simon/Data/Augsburg_large/patient_info.csv')

334it [00:00, 1422.13it/s]

[Errno 2] No such file or directory: '/home/simon/Data/Augsburg_large/preprocessed/PA000964/hip/IM000001'


617it [00:00, 1340.51it/s]

[Errno 2] No such file or directory: '/home/simon/Data/Augsburg_large/preprocessed/PA000917/hip/IM000001'


954it [00:00, 1349.26it/s]

[Errno 2] No such file or directory: '/home/simon/Data/Augsburg_large/preprocessed/PA000916/hip/IM000001'
